# 02b — Grouped Split (DECISION_LOG D015, supersedes D007)
**Why.** `03_duplicate_check.ipynb` showed that RealWaste photographs many items 2–3 times under neighbouring file numbers of the same class: 17 of 32 random triplets of consecutive files contained the same object, and 4 of 32 had that object in train *and* val/test. The image-level split of D007 therefore leaks the same items into validation and test.

**What.** The same 4752 images (read from the D007 `split.csv`, verified by SHA256) are grouped into blocks of `block_size` = 10 consecutive file numbers per class (`number // 10`). Whole blocks are assigned to train/val/test, stratified per class (≈70/15/15, seed 42). Leakage can remain only where re-shots straddle a block boundary; the notebook measures it.

**Output.** `02_Splits/split_grouped_v2.csv` and `02_Splits/split_grouped_v2_manifest.txt`. The D007 files are kept unchanged as the record. A guard refuses to overwrite an existing v2 file.

**After running.** Send the printed config block (hash + counts). Then `configs/config.yaml` is switched to the v2 split and `03` and `04` are re-run. A CPU runtime is enough; this notebook only reads CSV files.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/KNV-DSEB/realwaste-classification.git"
REPO_DIR = Path("/content/realwaste-classification")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print(subprocess.run(["git", "-C", str(REPO_DIR), "log", "-1", "--oneline"],
                     capture_output=True, text=True).stdout)

In [ ]:
import pandas as pd

from src.split import adjacent_pair_summary, grouped_stratified_split
from src.utils import get_paths, load_config, sha256_file

cfg = load_config(REPO_DIR / "configs" / "config.yaml")
paths = get_paths(cfg)

# Input: the D007 split, used only as the verified list of the 4752 valid images.
OLD_SPLIT = paths.drive_root / "02_Splits" / "split.csv"
OLD_SPLIT_SHA256 = "043e68e8028e402f2c2d04cdfa48e44b54c9ac6099981387f2859a0724aa4b1b"
assert sha256_file(OLD_SPLIT) == OLD_SPLIT_SHA256, "The D007 split.csv has changed; stop and investigate."
old = pd.read_csv(OLD_SPLIT)
assert len(old) == 4752 and old["filepath"].is_unique
print("D007 split:", old["split"].value_counts().to_dict())

In [ ]:
BLOCK_SIZE = cfg["split"]["block_size"]
new = grouped_stratified_split(
    old,
    block_size=BLOCK_SIZE,
    val_ratio=cfg["data"]["val_ratio"],
    test_ratio=cfg["data"]["test_ratio"],
    seed=cfg["project"]["seed"],
)

assert len(new) == len(old) and set(new["filepath"]) == set(old["filepath"])
assert (new.groupby(["class_name", "block"])["split"].nunique() == 1).all(), "a block spans several splits"

counts = new["split"].value_counts()
print("Counts:", {s: int(counts[s]) for s in ("train", "val", "test")})
print("Ratios:", {s: round(float(counts[s] / len(new)), 4) for s in ("train", "val", "test")})

distribution = pd.crosstab(new["class_name"], new["split"])[["train", "val", "test"]]
distribution["total"] = distribution.sum(axis=1)
for s in ("train", "val", "test"):
    distribution[f"{s}_pct"] = (distribution[s] / distribution["total"] * 100).round(1)
assert (distribution[["train", "val", "test"]] > 0).all().all(), "a class is missing from a split"
print(distribution.to_string())

In [ ]:
# Remaining leakage risk: same-class images whose file numbers differ by 1 or 2 but sit in different splits.
print("Image-level split (D007):")
print(adjacent_pair_summary(old).to_string())
print("\nGrouped split (D015):")
print(adjacent_pair_summary(new).to_string())

# Same-object pairs that crossed train vs val/test under D007 (03_duplicate_check.ipynb review)
KNOWN_PAIRS = [("Metal", 216, 217), ("Cardboard", 140, 142), ("Vegetation", 80, 82),
               ("Miscellaneous Trash", 408, 410), ("Plastic", 644, 646), ("Plastic", 15, 17)]
split_by_name = new.set_index(new["filepath"].str.rsplit("/", n=1).str[-1])["split"]
known = pd.DataFrame([
    {"pair": f"{c} {a} / {b}", "split_a": split_by_name[f"{c}_{a}.jpg"], "split_b": split_by_name[f"{c}_{b}.jpg"]}
    for c, a, b in KNOWN_PAIRS
])
known["same_split"] = known["split_a"] == known["split_b"]
print()
print(known.to_string(index=False))

In [ ]:
from datetime import date

NEW_SPLIT = paths.drive_root / "02_Splits" / "split_grouped_v2.csv"
MANIFEST = paths.drive_root / "02_Splits" / "split_grouped_v2_manifest.txt"
if NEW_SPLIT.exists():
    raise RuntimeError(f"{NEW_SPLIT} already exists; refusing to overwrite it. "
                       "Its SHA256 is recorded in configs/config.yaml.")

new.to_csv(NEW_SPLIT, index=False)
new_hash = sha256_file(NEW_SPLIT)
mapping_hash = sha256_file(paths.class_mapping_file)
assert mapping_hash == cfg["split"]["class_mapping_sha256"], "class_mapping.csv changed"

with open(MANIFEST, "w") as f:
    f.write("RealWaste grouped split (DECISION_LOG D015, supersedes D007)\n")
    f.write(f"Created: {date.today().isoformat()}\n")
    f.write(f"Seed: {cfg['project']['seed']}; block size: {BLOCK_SIZE}\n")
    for s in ("train", "val", "test"):
        f.write(f"{s}: {int(counts[s])}\n")
    f.write(f"Total: {len(new)}\n")
    f.write(f"split_grouped_v2.csv SHA256: {new_hash}\n")
    f.write(f"class_mapping.csv SHA256: {mapping_hash}\n")
print(open(MANIFEST).read())

print("---- send this block (configs/config.yaml, split section) ----")
print('  split_file: "02_Splits/split_grouped_v2.csv"')
print(f'  split_sha256: "{new_hash}"')
print("  expected_counts:")
for s in ("train", "val", "test"):
    print(f"    {s}: {int(counts[s])}")

## Next
1. Send the block printed above; `configs/config.yaml` is switched to `split_grouped_v2.csv` with its SHA256 and counts.
2. Re-run `03_duplicate_check.ipynb` (nearest-train check on the new split) and `04_pipeline_smoke_test.ipynb`.
3. Training starts only after both pass (Gate 2 on D015).